<a href="https://colab.research.google.com/github/Atin2001/GCI/blob/main/GCI2026W_%E7%AC%AC6%E5%9B%9E_%E6%95%99%E5%B8%AB%E3%81%82%E3%82%8A%E5%AD%A6%E7%BF%92_%E8%A7%A3%E7%AD%94_ipynb_%E3%81%AE%E3%82%B3%E3%83%94%E3%83%BC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 第6回 教師あり学習 — 練習問題・総合問題 解答


**注意**：表示スコアはscikit-learnのバージョンなどによって多少異なる場合があります。再現のため、分割の`random_state`と`test_size`を設問に合わせて固定しています。本ノートブックは上から順に実行してください。

## 共通で使うライブラリ

In [ ]:
import io
import requests
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer
from sklearn.tree import DecisionTreeClassifier

pd.set_option('display.max_columns', 30)

## 練習問題1・2


データセットの出典：Schlimmer, J. (1985). *Automobile* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5B01C [データセット1]


In [ ]:
# 本編と同じ列構成で自動車データを読み込む（練習問題1・2で共通使用）
auto_url = (
    'https://s3.ap-northeast-1.amazonaws.com/lecture-handouts.weblab/'
    '%E7%AC%AC6%E5%9B%9E/imports-85.data'
)
auto_columns = [
    'symboling', 'normalized-losses', 'make', 'fuel-type', 'aspiration',
    'num-of-doors', 'body-style', 'drive-wheels', 'engine-location',
    'wheel-base', 'length', 'width', 'height', 'curb-weight', 'engine-type',
    'num-of-cylinders', 'engine-size', 'fuel-system', 'bore', 'stroke',
    'compression-ratio', 'horsepower', 'peak-rpm', 'city-mpg',
    'highway-mpg', 'price'
]
response = requests.get(auto_url, timeout=30)
response.raise_for_status()
auto_raw = pd.read_csv(
    io.StringIO(response.content.decode('utf-8')),
    header=None,
    names=auto_columns
)
print('元データの形状:', auto_raw.shape)

#### <練習問題 1>

第2節で読み込んだ**元の自動車価格データ`auto`**を使います。`price`を目的変数、`length`と`engine-size`を説明変数として、重回帰モデルを構築してください。

必要な3列を取り出し、`'?'`などの欠損を処理したうえで、数値型に変換してください。`train_test_split(test_size=0.5, random_state=0)`で訓練・テストデータに分け、テストデータの決定係数（$R^2$）を求めましょう。

In [ ]:
# 使用する3列を選び、欠損を除去して数値型に揃える
auto_ex1 = (
    auto_raw[['price', 'length', 'engine-size']]
    .copy()
    .replace('?', np.nan)
    .apply(pd.to_numeric, errors='coerce')
    .dropna()
)

X1 = auto_ex1[['length', 'engine-size']]
y1 = auto_ex1['price']
X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.5, random_state=0
)

model1 = LinearRegression()
model1.fit(X1_train, y1_train)
print('使用件数:', len(auto_ex1))
print('決定係数 (train): {:.3f}'.format(model1.score(X1_train, y1_train)))
print('決定係数 (test) : {:.3f}'.format(model1.score(X1_test, y1_test)))
print('回帰係数:', dict(zip(X1.columns, model1.coef_)))

#### <練習問題 2>

同じ元データ`auto`から、練習問題1とは異なる説明変数を選んで`price`を予測してください。必要な欠損処理と型変換を行い、`train_test_split(test_size=0.5, random_state=0)`で分割して、テストデータの決定係数（$R^2$）を求めます。

選んだ説明変数によって結果がどう変わったか、その理由も考えてみましょう。

※2つのモデルの性能を公平に比較するには、評価に用いるデータの行と分割条件を揃えることも重要です。

**解答方針**：練習問題2では、本編で用いた `horsepower`・`width`・`height` を採用します。`horsepower` には欠損があるため、2モデルのスコアを公平に比べる際には**両方のモデルに使える共通の行だけ**を残し、訓練・テストの行も揃えます。この比較用データは練習問題1単独のデータと件数が異なる可能性があるので、練習問題1のスコアも共通データ上で改めて計算します。

In [ ]:
# 2種類のモデルで使用する全列の欠損を一度に処理して、共通の評価対象を作る
features_1 = ['length', 'engine-size']
features_2 = ['horsepower', 'width', 'height']
all_columns = ['price'] + features_1 + features_2

auto_compare = (
    auto_raw[all_columns]
    .copy()
    .replace('?', np.nan)
    .apply(pd.to_numeric, errors='coerce')
    .dropna()
)

train_idx, test_idx = train_test_split(
    auto_compare.index, test_size=0.5, random_state=0
)

comparison = []
for name, columns in [
    ('練習1の特徴量', features_1),
    ('練習2の特徴量', features_2),
]:
    model = LinearRegression()
    model.fit(auto_compare.loc[train_idx, columns], auto_compare.loc[train_idx, 'price'])
    comparison.append({
        'モデル': name,
        '説明変数': ', '.join(columns),
        '訓練R²': model.score(auto_compare.loc[train_idx, columns], auto_compare.loc[train_idx, 'price']),
        'テストR²': model.score(auto_compare.loc[test_idx, columns], auto_compare.loc[test_idx, 'price']),
    })

print('共通の評価対象件数:', len(auto_compare))
pd.DataFrame(comparison).round(3)

**考察例**：2種類の特徴量で予測性能が変わるのは、価格に関する情報量や特徴量間の相関が異なるためと考えられます。ただし、この結果は1回のランダム分割に基づくものであり、説明変数の**因果的な影響**を示すものではありません。

## 練習問題3・4
データセットについて
Breast Cancer Wisconsin (Diagnostic)データ
出典：Wolberg, W., Mangasarian, O., Street, N., & Street, W. (1993). *Breast Cancer Wisconsin (Diagnostic)* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5DW2B [データセット2]

`load_breast_cancer()` の30特徴量は、細胞診画像から**抽出・計算された数値特徴量**であり、生の画像ではありません。ここではscikit-learn同梱データを使用します。

#### <練習問題 3>

`sklearn.datasets`モジュールの`load_breast_cancer`関数を使って、**Breast Cancer Wisconsin (Diagnostic)** データセットを読み込み、目的変数を`cancer.target`、説明変数を`cancer.data`としてロジスティック回帰で予測モデルを構築してください。このとき、`train_test_split(test_size=0.3, random_state=0)`で訓練データとテストデータに分け、テストデータで正解率を求めてください。

※このデータセットは乳房腫瘤の穿刺吸引細胞診（FNA）の画像から計算した数値特徴量のデータであり、画像そのものを説明変数として扱う演習ではありません。出典は末尾に記載しています。

In [ ]:
cancer = load_breast_cancer()
X3_train, X3_test, y3_train, y3_test = train_test_split(
    cancer.data, cancer.target, test_size=0.3, random_state=0
)

# 標準化せずに学習する。収束しづらいため最大反復回数を増やす
model3 = LogisticRegression(max_iter=3000)
model3.fit(X3_train, y3_train)
score3_train = model3.score(X3_train, y3_train)
score3_test = model3.score(X3_test, y3_test)
print('正解率 (train): {:.3f}'.format(score3_train))
print('正解率 (test) : {:.3f}'.format(score3_test))

####  <練習問題 4>
<練習問題 3>と同じデータと分割条件を使い、訓練データで求めた平均・標準偏差を使って特徴量を標準化し、モデルを構築してください。そのうえで、練習問題3の結果と比較してください。

In [ ]:
# 練習問題3と「完全に同じ」訓練・テストデータを使用する
scaler4 = StandardScaler()
X4_train = scaler4.fit_transform(X3_train)  # 平均・標準偏差は訓練データのみ
X4_test = scaler4.transform(X3_test)       # テストデータへのfitは禁止

model4 = LogisticRegression(max_iter=3000)
model4.fit(X4_train, y3_train)
score4_train = model4.score(X4_train, y3_train)
score4_test = model4.score(X4_test, y3_test)

pd.DataFrame([
    {'モデル': '標準化なし（練習3）', '訓練正解率': score3_train, 'テスト正解率': score3_test},
    {'モデル': '標準化あり（練習4）', '訓練正解率': score4_train, 'テスト正解率': score4_test}
]).round(3)

**考察例**：標準化は数値スケールを揃えて最適化を安定させます。scikit-learnのロジスティック回帰では正則化の効き方にも影響するため、スコア差を「収束速度だけ」の効果と解釈しないようにしましょう。標準化の`fit`は訓練データだけで行います。

#### <練習問題 5>

`sklearn.datasets`モジュールの`load_breast_cancer`関数から乳がんデータを読み込み、目的変数を`cancer.target`、説明変数を`cancer.data`として決定木モデルを構築してください。

1. `max_depth=2`, `4`, `None`を試し、訓練スコアとテストスコアを比較してください。  
2. どの設定で過学習が起きていそうか考えてください。  
3. 余裕があれば`criterion='entropy'`も試し、標準設定の`gini`と比較してください。

In [ ]:
# 本編の練習問題5：決定木の深さと不純度の指標を比較する
cancer5 = load_breast_cancer()
X5_train, X5_test, y5_train, y5_test = train_test_split(
    cancer5.data, cancer5.target, test_size=0.3, random_state=0
)

results5 = []
for criterion in ['gini', 'entropy']:
    for depth in [2, 4, None]:
        tree5 = DecisionTreeClassifier(
            criterion=criterion, max_depth=depth, random_state=0
        )
        tree5.fit(X5_train, y5_train)
        train_acc = tree5.score(X5_train, y5_train)
        test_acc = tree5.score(X5_test, y5_test)
        results5.append({
            'criterion': criterion,
            'max_depth': str(depth),
            '訓練正解率': train_acc,
            'テスト正解率': test_acc,
            '訓練−テスト差': train_acc - test_acc,
        })

pd.DataFrame(results5).round(3)

**考察の視点**：深い木は訓練データに適合しやすい一方、テスト正解率が同じように上がるとは限りません。訓練−テスト差が大きい場合は過学習の可能性があります。ただし、1回の分割だけでは断定できません。最終的な深さや`criterion`の選択は、テストデータを繰り返し参照せず、検証データや交差検証で行います。

---

# 総合問題

### ■ 総合問題 1 教師あり学習の用語

次の用語について、それぞれの役割や意味を自分の言葉で説明してください。

- 教師あり学習
- 回帰 / 分類
- 説明変数 / 目的変数
- 訓練データ / テストデータ
- 過学習 / 汎化性能
- 重回帰
- ロジスティック回帰
- 標準化
- 正則化
- Ridge / Lasso
- 決定木
- 不純度

**発展**：エントロピー、情報利得、k-NN、SVMについても調べてみましょう。

**解答例（概要）**

- **教師あり学習**：説明変数と正解となる目的変数の組から、未知データの予測に利用する規則性を学ぶ方法。
- **回帰 / 分類**：回帰は連続的な数値を、分類はカテゴリを予測する問題。
- **説明変数 / 目的変数**：予測に入力する情報が説明変数、予測の対象が目的変数。
- **訓練データ / テストデータ**：学習に使うデータと、最終的な予測性能を評価するために学習から分離したデータ。
- **過学習 / 汎化性能**：訓練データに合わせすぎて未知データの性能が下がる現象と、未知データでも適切に予測できる能力。
- **重回帰**：複数の説明変数から連続値を予測する線形回帰。係数から因果関係は直ちには結論できない。
- **ロジスティック回帰**：クラスに属する確率をモデル化して分類を行う手法。
- **標準化**：訓練データの平均と標準偏差を用いて特徴量のスケールを揃える前処理。
- **正則化**：係数の大きさなどにペナルティを加え、モデルの過度な複雑さを抑える仕組み。
- **Ridge / Lasso**：それぞれ主にL2 / L1正則化を用いる回帰手法。Lassoは係数を0にすることがある。
- **決定木**：特徴量への条件分岐を繰り返して予測するモデル。
- **不純度**：分類木のノード内に異なるクラスがどの程度混ざっているかを表す尺度。

**発展用語**：エントロピーは不純度の尺度の一つ、情報利得は分割による不純度の減少を表す。k-NNは近傍のデータから予測し、SVMはクラスを分ける境界を学習する手法。

### ■ 総合問題 2 決定木

練習問題で利用した乳がんデータについて、決定木を使って予測と評価を行ってください。`max_depth`を複数試し、訓練データとテストデータのスコアから、モデルの複雑さと過学習の関係を考察してください。

In [ ]:
# 総合問題2：決定木の複雑さ・予測・評価
from sklearn.metrics import confusion_matrix

cancer_total2 = load_breast_cancer()
X_t2_train, X_t2_test, y_t2_train, y_t2_test = train_test_split(
    cancer_total2.data, cancer_total2.target,
    test_size=0.3, random_state=0
)

rows_t2 = []
for depth in [2, 4, None]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0)
    tree.fit(X_t2_train, y_t2_train)
    rows_t2.append({
        'max_depth': str(depth),
        '訓練正解率': tree.score(X_t2_train, y_t2_train),
        'テスト正解率': tree.score(X_t2_test, y_t2_test),
    })
display(pd.DataFrame(rows_t2).round(3))

# 深さ4を例にした「予測の確認」。テスト結果を用いて選んだ深さではない
example_tree = DecisionTreeClassifier(max_depth=4, random_state=0)
example_tree.fit(X_t2_train, y_t2_train)
y_t2_pred = example_tree.predict(X_t2_test)
print('混同行列（行=真のクラス、列=予測クラス）:')
print(confusion_matrix(y_t2_test, y_t2_pred, labels=[0, 1]))

**考察例**：深さを増やすと訓練正解率は上がりやすくなりますが、未知データでも改善する保証はありません。乳がんデータは`target=0`が悪性、`target=1`が良性です。混同行列では、どの種類の誤判定が起きたかも確認できます。実際に重要な誤判定の種類は、利用目的に応じて評価する必要があります。

### ■ 総合問題 3 モデルを比較する

同じ分類データに対して、ロジスティック回帰と決定木を適用し、テストデータの結果を比較してみましょう。

モデルによって得意・不得意があり、**どの問題でも必ず最良になる単一の学習アルゴリズムがあるわけではありません**。この実務上の考え方は、機械学習で知られる「ノーフリーランチ」の議論とも関連しています。ただし、定理の厳密な主張とは区別し、ここでは「データや目的に応じてモデルを比較・評価することが重要」と理解してください。

In [ ]:
# 総合問題3：同じ分割のデータでロジスティック回帰と決定木を比較
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

cancer_total3 = load_breast_cancer()
X_t3_train, X_t3_test, y_t3_train, y_t3_test = train_test_split(
    cancer_total3.data, cancer_total3.target,
    test_size=0.3, random_state=0
)

models_t3 = {
    # Pipelineにより標準化を訓練データのみに適用する
    'ロジスティック回帰': make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
    # 決定木はスケールの大小による前処理が基本的に不要
    '決定木（深さ4）': DecisionTreeClassifier(max_depth=4, random_state=0),
}

rows_t3 = []
for name, model in models_t3.items():
    model.fit(X_t3_train, y_t3_train)
    rows_t3.append({
        'モデル': name,
        '訓練正解率': model.score(X_t3_train, y_t3_train),
        'テスト正解率': accuracy_score(y_t3_test, model.predict(X_t3_test)),
    })
pd.DataFrame(rows_t3).round(3)

**考察例**：同じデータ分割でも、モデルの仕組みや前処理により結果は異なります。ここでは代表的な固定設定を比較しています。モデルの優劣を一般化するには、交差検証や別データでの評価、重要な誤判定、解釈性、実行コストなども考慮します。テスト正解率が高いモデルをこの1回の結果だけで「常に優れている」とは判断しません。

## 使用データセット・参考文献について

　本教材では、UCI Machine Learning Repositoryで公開されている以下の外部データセットを使用しています。（出典・ライセンス情報の最終確認日：2026年10月31日）


[データセット1] **Automobile**（練習問題1・2）  
   Schlimmer, J. (1985). *Automobile* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5B01C  
   UCI公式ページ：https://archive.ics.uci.edu/dataset/10/automobile  
   元データは1985年のWard's Automotive Yearbookに由来します。本解答では講義用S3の`imports-85.data`を取得し、欠損処理・数値化・列選択・訓練/テスト分割を行います。

[データセット2] **Breast Cancer Wisconsin (Diagnostic)**（練習問題3〜5・総合問題2・3）  
   Wolberg, W., Mangasarian, O., Street, N., & Street, W. (1993). *Breast Cancer Wisconsin (Diagnostic)* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5DW2B  
   UCI公式ページ：https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic  
   配布・読み込み：scikit-learn [`load_breast_cancer()`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html)。モデル用に学習/テスト分割・標準化等を行っています。
  


**ライセンスおよび利用に関する注記**

[データセット1][データセット2]のデータセットはCreative Commons Attribution 4.0 International（CC BY 4.0）ライセンスの下で公開されています。

- ライセンス詳細：https://creativecommons.org/licenses/by/4.0/

**データファイルの提供方法**

[データセット1]のデータセットは学習の円滑化のため学習の円滑化のため以下のリンクより直接取得しています。
- https://s3.ap-northeast-1.amazonaws.com/lecture-handouts.weblab/%E7%AC%AC6%E5%9B%9E/imports-85.data'

**関連論文・参考文献**

[参考文献]
- 塚本邦尊、山田典一、大澤文孝『東京大学のデータサイエンティスト育成講座』（マイナビ出版、2019年）
